# MACE MLIP minimal CPU training script walkthrough

This notebook explains `../examples/mace_mlip_training_cpu_minimal.sh`, a small local training example for a VS Code terminal or ordinary shell. It trains a compact MACE model on CPU using extended XYZ files with `ref_energy` and `ref_forces` labels.

The longer foundation-model workflow is now kept as `../examples/mace_mlip_training_advanced.sh`. Use that advanced script when you need multihead fine-tuning, replay data, GPU execution, or distributed training.

## Documentation referenced

The minimal example uses the standard MACE training command-line interface. The advanced workflow uses the foundation-model and distributed-training features listed below.

- Training guide: https://mace-docs.readthedocs.io/en/latest/guide/training.html
- Fine-tuning foundation models: https://mace-docs.readthedocs.io/en/latest/guide/finetuning.html
- Multihead replay fine-tuning: https://mace-docs.readthedocs.io/en/latest/guide/multihead_finetuning.html
- Multi-GPU training: https://mace-docs.readthedocs.io/en/latest/guide/multigpu.html
- Foundation models: https://mace-docs.readthedocs.io/en/latest/guide/foundation_models.html
- MACE foundation-model releases and replay data: https://github.com/ACEsuit/mace-foundations/releases

In [1]:
from pathlib import Path

script_path = Path("../examples/mace_mlip_training_cpu_minimal.sh")
print(script_path.resolve())
print(script_path.read_text())

C:\Users\c1528354\GitHub\chem4energy-2026\chem4energy-2026\examples\mace_mlip_training_cpu_minimal.sh
#!/usr/bin/env bash
set -euo pipefail

# Minimal local CPU training example for a VS Code terminal.
#
# Run from the repository root after activating your Python environment.
# The input files should be extended XYZ files with ref_energy/ref_forces labels.
#
#   TRAIN_FILE=data/train.extxyz TEST_FILE=data/test.extxyz ./examples/mace_mlip_training_cpu_minimal.sh

RUN_NAME="${RUN_NAME:-mace_cpu_minimal}"
TRAIN_FILE="${TRAIN_FILE:-data/train.extxyz}"
TEST_FILE="${TEST_FILE:-data/test.extxyz}"

python -m mace.cli.run_train \
    --name="$RUN_NAME" \
    --train_file="$TRAIN_FILE" \
    --valid_fraction=0.05 \
    --test_file="$TEST_FILE" \
    --E0s="average" \
    --model="MACE" \
    --num_interactions=2 \
    --num_channels=32 \
    --max_L=1 \
    --correlation=2 \
    --r_max=5.0 \
    --energy_key="ref_energy" \
    --forces_key="ref_forces" \
    --energy_weight=10 \
    --forces_we

## Workflow shape

The minimal script has three parts:

1. Set `RUN_NAME`, `TRAIN_FILE`, and `TEST_FILE`, then find a Python interpreter from `PYTHON`, `python3`, or `python`.
2. Launch MACE directly with `python -m mace.cli.run_train` and a deliberately small CPU model.
3. Use restartable MACE outputs with `--restart_latest` and write CPU-loadable checkpoints with `--save_cpu`.

This is intended as the first runnable check for data formatting and local MACE installation. It leaves scheduler setup, GPU launchers, foundation models, replay data, and long training runs to the advanced script.

## Preparing an extended XYZ training file from `.traj`

The helper script `../examples/prepare_training_set.py` converts an ASE trajectory file into an extended XYZ file suitable for the MACE training scripts. It reads the structures from `.traj`, extracts stored energies and forces, then writes them as `ref_energy` and `ref_forces`.

From the repository root, convert all frames in a trajectory with:

```bash
python examples/prepare_training_set.py opt.traj training.extxyz
```

If your terminal only has `python3`, use:

```bash
python3 examples/prepare_training_set.py opt.traj training.extxyz
```

The output filename is optional. Without it, the script changes the suffix of the input file:

```bash
python examples/prepare_training_set.py opt.traj
```

This writes `opt.extxyz`.

You can select frames using ASE's normal index syntax. For example, the last frame only:

```bash
python examples/prepare_training_set.py opt.traj final_structure.extxyz --index -1
```

The first 50 frames:

```bash
python examples/prepare_training_set.py opt.traj training.extxyz --index 0:50
```

The generated file can then be passed to the minimal CPU example:

```bash
TRAIN_FILE=training.extxyz TEST_FILE=test.extxyz ./examples/mace_mlip_training_cpu_minimal.sh
```

If the source trajectory already stores energies or forces under project-specific names, use `--source-energy-key` and `--source-forces-key` to tell the converter where to read them from.

## Input and label choices

| Script setting | MACE flag | Meaning |
|---|---|---|
| `RUN_NAME` | `--name` | Name used by MACE for logs, checkpoints, and output files. Defaults to `mace_cpu_minimal`. |
| `TRAIN_FILE` | `--train_file` | Extended XYZ file containing the training structures. Defaults to `data/train.extxyz`. |
| `TEST_FILE` | `--test_file` | Independent test set evaluated after training. Defaults to `data/test.extxyz`. |
| fixed `0.05` | `--valid_fraction` | Holds out 5 percent of `TRAIN_FILE` for validation. |
| fixed `average` | `--E0s` | Lets MACE estimate isolated-atom reference energies from the dataset for this small example. |
| fixed `ref_energy` | `--energy_key` | Name of the energy field in each XYZ configuration. |
| fixed `ref_forces` | `--forces_key` | Name of the forces array in each XYZ configuration. |

The MACE training guide describes `--train_file`, validation options, and `--test_file` as the standard data split controls. It also notes that energies and forces default to keys named `energy` and `forces`, so this example explicitly points MACE at `ref_energy` and `ref_forces`.

Those `ref_` names keep DFT reference labels separate from calculator-generated ASE values that may also appear in XYZ metadata. For serious production training, replace `--E0s=average` with reference energies appropriate for the chemistry and reference method, or start from the advanced workflow.

## Model size, loss, and optimiser flags

| MACE flag | Value in the minimal script | Meaning |
|---|---|---|
| `--model` | `MACE` | Trains a standard MACE model from scratch. |
| `--num_interactions` | `2` | Uses a shallow message-passing stack for a quick local run. |
| `--num_channels` | `32` | Keeps the model small enough for CPU testing. |
| `--max_L` | `1` | Limits angular features for speed. |
| `--correlation` | `2` | Keeps body-order/correlation settings modest. |
| `--r_max` | `5.0` | Neighbour cutoff radius. |
| `--energy_weight` | `10` | Weight of the energy term in the loss. |
| `--forces_weight` | `100` | Weight of the force term in the loss. Force fitting commonly receives a larger weight for MLIPs. |
| `--batch_size` and `--valid_batch_size` | `1` | Processes one configuration at a time to avoid memory surprises. |
| `--max_num_epochs` | `5` | Runs only a short smoke test. |
| `--eval_interval` | `1` | Reports validation metrics every epoch. |
| `--ema` | enabled | Uses exponential moving average of weights for training stability. |
| `--amsgrad` | enabled | Uses the AMSGrad variant of Adam. |
| `--error_table` | `PerAtomMAE` | Prints per-atom mean absolute errors. |

These values are intentionally small. The goal is to prove that the data, labels, and MACE installation work before spending time on a larger model.

## Runtime flags

| Script setting | MACE flag or behaviour | Meaning |
|---|---|---|
| `PYTHON` | launcher | Optional path or command for the Python interpreter. If unset, the script tries `python3` and then `python`. |
| fixed `cpu` | `--device` | Runs on CPU so the example works without a GPU. |
| fixed `float32` | `--default_dtype` | Uses single precision to keep the CPU run lighter. |
| fixed `123` | `--seed` | Sets the random seed for reproducibility. |
| enabled | `--restart_latest` | Restarts from the latest checkpoint if one exists for the same run name. |
| enabled | `--save_cpu` | Saves a checkpoint that can be loaded on CPU. |

The minimal script does not use `torchrun`, `--distributed`, CUDA, replay datasets, or foundation-model flags. Those belong in `mace_mlip_training_advanced.sh`.

In [ ]:
example_commands = [
    "TRAIN_FILE=data/train.extxyz TEST_FILE=data/test.extxyz ./examples/mace_mlip_training_cpu_minimal.sh",
    "PYTHON=.venv/bin/python TRAIN_FILE=data/train.extxyz TEST_FILE=data/test.extxyz ./examples/mace_mlip_training_cpu_minimal.sh",
    "RUN_NAME=mace_tinio3_smoke TRAIN_FILE=training.extxyz TEST_FILE=test.extxyz ./examples/mace_mlip_training_cpu_minimal.sh",
]

for command in example_commands:
    print(command)

## Practical checks before training

- Confirm the training and test files are extended XYZ files and contain `ref_energy` and `ref_forces`.
- Keep the test set independent of the structures used for training and validation.
- Start with a small number of frames if you only want to check that MACE can read the data.
- Change `RUN_NAME` when you want a fresh run instead of restarting from existing checkpoints.
- Treat the minimal CPU script as a smoke test. Move to the advanced script when you need a foundation model, GPU training, replay data, explicit `E0s`, or longer convergence checks.

## Advanced workflow

The advanced workflow lives in `../examples/mace_mlip_training_advanced.sh`. It keeps the scheduler-neutral foundation-model fine-tuning setup with replay data, user-configurable validation files, GPU execution, optional single-node multi-GPU launch, logging through `tee`, and longer training defaults.

From the repository root, run it only after the minimal example has verified the input data:

```bash
TRAIN_FILE=data/train.xyz TEST_FILE=data/test.xyz ./examples/mace_mlip_training_advanced.sh
```

For a single-node multi-GPU run:

```bash
NPROC_PER_NODE=4 TRAIN_FILE=data/train.xyz TEST_FILE=data/test.xyz ./examples/mace_mlip_training_advanced.sh
```

## Where outputs go

The minimal script does not change directory or copy files to scratch. MACE writes its normal outputs relative to the directory from which you run the script.

Because the script includes `--restart_latest`, a stopped or interrupted training job can be restarted by running the same command again from the same folder with the same `RUN_NAME`. MACE will look for the latest compatible checkpoint and continue from it.

If you want a completely fresh training run, change `RUN_NAME` or move the previous outputs elsewhere before restarting.